# Surface Code Decoding

In this notebook, we aim to develop a minimum-weight decoder for nine-qubit rotated surface code. We will begin with creating binary representations (vectors) of stabilizers and logical operators. Then, we will build decoding graphs and evaluate recovery for every $Z$-error pattern.

Now, the current model uses $Z$-only errors and syndrome measurements. Each qubit independently has a $Z$-error with probability $p$. Next, we are going to reveal circuit-based syndrome extraction in Qiskit.

**Content**

1. Define the code and binary representation
2. Identify stabilizers and logical operators
3. Construct the decoding graph
4. Generate all Z-errors
5. Build the minimum-weight decoder table
6. Evaluate recovery and logical failure probability

## Define the code and binary representation

We are going to work on nine-qubit rotated surface code. The data qubits $q_i$ are placed on a physical patch as follows:

\\begin{matrix}q_0&q_1&q_2\\q_3&q_4&q_5\\q_6&q_7&q_8\end{matrix}

Suppose there is a $Z$-error on a qubit $j$. It means that the Pauli-$Z$ operator acts on the qubit $q_j$. In this case, we represent the error by an error vector $e$ such that $e_j = 1$. For example, suppose that there is an error string `ZZIZIIIII`. Then, the binary representation is $e = (\begin{matrix}1 & 1 & 0&1&0&0&0&0&0\end{matrix})$

$X$ and $Z$-type stabilizer generators are represented by matrices $H_x$ and $H_z$. Multiplying of same type operators corresponds to adding support vectors modulo two. A $Z$- and $X$-error *syndrome* is computed as follows:

$$ s_z = H_X e \text{ (mod 2)} $$
$$ s_x = H_Z e \text{ (mod 2)} $$

Let us remind of syndromes Given a quantum state $\ket{\psi}$, assume that $E$ is error and $S$ denotes the stabilizer. Then,

$$ ES\ket{\psi} = (\pm 1) SE \ket{\psi}$$ Since $S\ket{\psi} = \ket{\psi}$,
$$ ES\ket{\psi} = E \ket{\psi} = (\pm 1) SE \ket{\psi}$$ So, $\pm 1$ gives eigenvalues of $S$. We call these eigenvalues *syndrome*. So the syndrome vectors, $s_x$ and $s_z$, consist of these eigenvalues. However, we store $0$ and $1$ in the vectors instead of $\pm 1$. A syndrome bit of $1$ corresponds to the eigenvalue $-1$ while $0$ records the eigenvalue $+1$.




Given the $X$- and $Z$-type stabilisers $$ S_X = <XXIIIIIII, IXXIXXIII, IIIXXIXXI, IIIIIIIXX> $$
$$ S_Z = <ZZIZZIIII, IIZIIZIII, IIIZIIZII, IIIIZZIZZ> $$

let us view the stabiliser check matrices $H_X$ and $H_Z$:

In [2]:
import numpy as np

H_x = np.array([
    [1, 1, 0, 0, 0, 0, 0, 0, 0], # XXIIIIIII
    [0, 1, 1, 0, 1, 1, 0, 0, 0], # IXXIXXIII
    [0, 0, 0, 1, 1, 0, 1, 1, 0], # IIIXXIXXI
    [0, 0, 0, 0, 0, 0, 0, 1, 1], # IIIIIIIXX
], dtype=int)

H_z = np.array([
    [1, 1, 0, 1, 1, 0, 0, 0, 0], # ZZIZZIIII
    [0, 0, 1, 0, 0, 1, 0, 0, 0], # IIZIIZIII
    [0, 0, 0, 1, 0, 0, 1, 0, 0], # IIIZIIZII
    [0, 0, 0, 0, 1, 1, 0, 1, 1], # IIIIZZIZZ
], dtype=int)

All X-type and Z-type generators commute.


Note that inner product of two vectors of $H_x$ and $H_z$ is a mathematical representation of product of two stabilisers consisting of Pauli strings. Remind that the number of operators that overlap at the same position is even, then the two operators commute. Otherwise, they anticommute with each other. In the mathematical way, we check the commutation looking at the inner product modulo $2$ of the two representations. If the product is $0$, then we understand that the number of the number of errors at the same position is even. The inner product result $1$ says that the number of overlapping operators is odd. Thus, we can express that

if $ <g_X, g_Z> = 1 $, then two generators $g_X$ and $g_Z$ anticommute

if $ <g_X, g_Z> = 0 $, then two generators $g_X$ and $g_Z$ commute

We can check all type generators through the check matrices $H_X$ and $H_Z$:

In [3]:
# checking commutation
assert np.all((H_x @ H_z.T) % 2 == 0)
print("All X-type and Z-type generators commute.")

All X-type and Z-type generators commute.
